# 01 🎛️ 予測のツマミ: temperature / probability_mode / top_k_edges

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagioneko/cse-frog/blob/main/notebooks/01_prediction_knobs.ipynb)

> 解説は [HANDBOOK](../docs/HANDBOOK.md) と [CONFIG_GUIDE](../docs/CONFIG_GUIDE.md)、全設定は [REFERENCE](../docs/REFERENCE.md) にあります。

In [1]:
# 🐸 の準備(Colab では最初に1回だけインストールします)
try:
    import cse
except ImportError:
    %pip install -q cse-frog
from cse import Frog, END

## 1. Temperature Ladder
2つの続きを 7:3 で覚えさせ、`temperature` だけを変えます。

In [2]:
data = [["右", "下"]] * 7 + [["右", "上"]] * 3
for t in [0.05, 0.2, 0.8, 2.0, 10.0]:
    f = Frog(temperature=t).learn(data)
    print(f"temperature={t:<5} {f.top(['右'])}")

temperature=0.05  [('下', 0.9999999999999742), ('上', 2.5794025346876066e-14)]
temperature=0.2   [('下', 0.9995994047739734), ('上', 0.0004005952260265635)]
temperature=0.8   [('下', 0.8760494229318633), ('上', 0.12395057706813667)]
temperature=2.0   [('下', 0.6861574810795763), ('上', 0.31384251892042375)]
temperature=10.0  [('下', 0.5390312186529266), ('上', 0.46096878134707336)]


低いほど1位が確定的に、高いほど平らになります。**点数(`scores`)は同じ**で、確率への変換だけが変わっています。

In [3]:
print(Frog(temperature=0.05).learn(data).scores(["右"]))
print(Frog(temperature=10.0).learn(data).scores(["右"]))

{'下': 2.749323789278667, '上': 1.1848921140034987}
{'下': 2.749323789278667, '上': 1.1848921140034987}


## 2. softmax と linear
`linear` は点数の比をそのまま確率にします(`temperature` は使われません)。

In [4]:
for mode in ["softmax", "linear"]:
    f = Frog(probability_mode=mode).learn(data)
    print(f"{mode:<8} {f.top(['右'])}")
f = Frog(probability_mode="linear", temperature=10.0).learn(data)
print("linear + temperature=10:", f.top(["右"]), "← 変わらない")

softmax  [('下', 0.8760494229318633), ('上', 0.12395057706813667)]
linear   [('下', 0.6988238207732859), ('上', 0.3011761792267142)]
linear + temperature=10: [('下', 0.6988238207732859), ('上', 0.3011761792267142)] ← 変わらない


## 3. top_k_edges: 候補を何個まで確率にするか
「A」の後に5種類の続きを、回数を変えて覚えさせます。点数が正の候補が **K個を超えたとき** だけ差が出ます。

In [5]:
data5 = [["A", "B"]] * 5 + [["A", "C"]] * 4 + [["A", "D"]] * 3 + [["A", "E"]] * 2 + [["A", "F"]] * 1
for k in [1, 2, 3, 5, 10]:
    f = Frog(top_k_edges=k).learn(data5)
    probs = f.probabilities(["A"])
    print(f"top_k_edges={k:<3} 候補{len(probs)}個: " + ", ".join(f"{s}={p:.2f}" for s, p in f.top(['A'], k=10)))

top_k_edges=1   候補1個: B=1.00
top_k_edges=2   候補2個: B=0.61, C=0.39
top_k_edges=3   候補3個: B=0.48, C=0.31, D=0.20
top_k_edges=5   候補5個: B=0.40, C=0.26, D=0.17, E=0.11, F=0.07
top_k_edges=10  候補5個: B=0.40, C=0.26, D=0.17, E=0.11, F=0.07


`top_k_edges` には **活性を広げる本数** というもう1つの仕事もあります。そちらは `history_boost > 0` のときだけ予測に届きます(**04** で確かめます)。